Объединение поездок и погоды

Входные данные:
- `rides_clean.csv` – очищенные поездки (id, start_date, end_date, distance, promo, start_district, end_district, duration, price, start_fee, total_cost)  
- `weather_clean.csv` – очищенная погода (datetime, temperature, precipitation_total, wind_gust, wind_speed, cloud_cover_total, sunshine_duration)  

In [5]:
import pandas as pd
import numpy as np

rides = pd.read_csv('rides_clean.csv', parse_dates=['start_date', 'end_date'])
weather = pd.read_csv('weather_clean.csv', parse_dates=['datetime'])

rides['start_hour'] = rides['start_date'].dt.floor('h')

weather['hour'] = weather['datetime'].dt.floor('h')
weather = weather.drop_duplicates('hour')

df = rides.merge(weather, left_on='start_hour', right_on='hour', how='left')

missing = df['temperature'].isna().sum()
if missing:
    print(f"Удаляем {missing} поездок без погоды ({missing/len(df)*100:.2f}%)")
    df = df.dropna(subset=['temperature'])

df['speed'] = (df['distance'] / 1000) / (df['duration'] / 60)
df['hour_of_day'] = df['start_date'].dt.hour
df['day_of_week'] = df['start_date'].dt.dayofweek   # 0=пн
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['start_date'].dt.month

len0 = len(df)
df = df[(df['distance'] >= 100) & (df['distance'] <= 15000)]
df = df[(df['duration'] >= 1) & (df['duration'] <= 120)]
df = df[(df['speed'] >= 3) & (df['speed'] <= 30)]
df = df[df['precipitation_total'] >= 0]
df = df[(df['temperature'] >= -5) & (df['temperature'] <= 40)]
print(f"Удалено {len0 - len(df)} строк ({100*(len0 - len(df))/len0:.2f}%)")

df.to_csv('rides_weather_clean.csv', index=False)
print(f"Готово. Размер: {df.shape}")

Удаляем 3187 поездок без погоды (3.46%)
Удалено 39 строк (0.04%)
Готово. Размер: (88896, 27)
